In [1]:
import torch
import numpy as np


In [2]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    else:
        return torch.device("cpu")

get_device = get_device()
get_device

device(type='cuda')

In [3]:
import pandas as pd

try: # KAGGLE
    df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
    test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
except: 
    try: # COLAB
        df = pd.read_csv('train.csv')
        test_df = pd.read_csv('test.csv')
    except: # LOCAL
        df = pd.read_csv('../data/train.csv')
        test_df = pd.read_csv('../data/test.csv')
        
df.shape, test_df.shape

((2000, 8), (500, 7))

In [4]:
df.columns

Index(['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'], dtype='object')

## Dataset Analysis

In [5]:
print("="*70)
print("📊 DATASET ANALYSIS")
print("="*70)
print(f"\nTraining set shape: {df.shape}")
print(f"Test set shape: {test_df.shape}")
print(f"\nTraining columns: {df.columns.tolist()}")
print(f"Test columns: {test_df.columns.tolist()}")

# Check answer distribution
print("\n📈 ANSWER DISTRIBUTION IN TRAINING DATA")
print("-" * 70)
answer_dist = df['answer'].value_counts().sort_index()
for ans, count in answer_dist.items():
    pct = (count / len(df)) * 100
    print(f"  {ans}: {count:4d} ({pct:6.2f}%)")

# Check for missing values
print("\n✓ Missing values check:")
print(f"  Training: {df.isnull().sum().sum()} missing values")
print(f"  Test: {test_df.isnull().sum().sum()} missing values")

# Show sample data
print("\n📝 Sample training data (first row):")
print(f"  ID: {df.iloc[0]['id']}")
print(f"  Prompt: {df.iloc[0]['prompt'][:80]}...")
print(f"  Answer: {df.iloc[0]['answer']}")
print(f"  Options: A, B, C, D, E")

📊 DATASET ANALYSIS

Training set shape: (2000, 8)
Test set shape: (500, 7)

Training columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
Test columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E']

📈 ANSWER DISTRIBUTION IN TRAINING DATA
----------------------------------------------------------------------
  A:  369 ( 18.45%)
  B:  490 ( 24.50%)
  C:  459 ( 22.95%)
  D:  358 ( 17.90%)
  E:  324 ( 16.20%)

✓ Missing values check:
  Training: 0 missing values
  Test: 0 missing values

📝 Sample training data (first row):
  ID: 1
  Prompt: Pick the best possible answer: What is Martin Heidegger's view on the relationsh...
  Answer: B
  Options: A, B, C, D, E


set seed

In [20]:
from transformers import set_seed

set_seed(42)

In [7]:
print("\n" + "="*70)
print("🔐 LABEL MAPPING VERIFICATION")
print("="*70)
print(f"OPTIONS: {OPTIONS}")
print(f"NUM_CHOICES: {NUM_CHOICES}")
print("\nLabel to Index Mapping (used in training):")
label2id = {opt: i for i, opt in enumerate(OPTIONS)}
id2label = {v: k for k, v in label2id.items()}
print(f"  {label2id}")
print(f"Reverse Mapping (Index to Label):")
print(f"  {id2label}")

# Verify mapping matches training data
print("\nTraining answer distribution by index:")
answer_indices = df['answer'].map(label2id)
for i in range(NUM_CHOICES):
    count = (answer_indices == i).sum()
    pct = 100 * count / len(df)
    print(f"  Index {i} ({OPTIONS[i]}): {count} ({pct:.2f}%)")


🔐 LABEL MAPPING VERIFICATION
OPTIONS: ['A', 'B', 'C', 'D', 'E']
NUM_CHOICES: 5

Label to Index Mapping (used in training):
  {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
Reverse Mapping (Index to Label):
  {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

Training answer distribution by index:
  Index 0 (A): 369 (18.45%)
  Index 1 (B): 490 (24.50%)
  Index 2 (C): 459 (22.95%)
  Index 3 (D): 358 (17.90%)
  Index 4 (E): 324 (16.20%)


# Config

In [17]:
MODEL_NAME = "microsoft/deberta-v3-base"
OPTIONS = ["A", "B", "C", "D", "E"]
NUM_CHOICES = len(OPTIONS)
BATCH_SIZE = 100
EPOCHS = 3

---

In [23]:
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice, Trainer, TrainingArguments
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load model directly in half precision on GPU to reduce memory usage.
use_fp16 = torch.cuda.is_available()
if not use_fp16:
    raise RuntimeError("GPU is required for this notebook setup; float32 CPU training is disabled due to memory limits.")

model = AutoModelForMultipleChoice.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    low_cpu_mem_usage=True,
)

label2id = {opt: i for i, opt in enumerate(OPTIONS)}
id2label = {v: k for k, v in label2id.items()}

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weigh

In [24]:
print("Model parameter dtype:", next(model.parameters()).dtype)
print("CUDA available:", torch.cuda.is_available())

Model parameter dtype: torch.float16
CUDA available: True


# Preprocess function

In [26]:
def preprocess_function(dataset):
    import itertools
    options = ['A', 'B', 'C', 'D', 'E']

    batch_size = len(dataset['prompt'])
    first_sentences = [[question] * 5 for question in dataset['prompt']]
    second_sentences = [[dataset[option][i] for option in options] for i in range(batch_size)]

    first_sentences = list(itertools.chain.from_iterable(first_sentences))
    second_sentences = list(itertools.chain.from_iterable(second_sentences))

    tokenized = tokenizer(
        first_sentences,
        second_sentences,
        padding='max_length',
        truncation=True,
        max_length=256,
    )

    output = {
        key: [values[i:i + 5] for i in range(0, len(values), 5)]
        for key, values in tokenized.items()
    }

    if 'answer' in dataset and dataset['answer'] is not None:
        mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
        # CrossEntropyLoss expects int64 labels.
        output['labels'] = np.array([mapping[ans] for ans in dataset['answer']], dtype=np.int64)

    return output

## Check preprocessing function

In [20]:
# Test preprocessing on a single sample
print("="*70)
print("🧪 TESTING PREPROCESSING FUNCTION")
print("="*70)

# Get one sample
sample = df.iloc[0:1].to_dict('list')
print(f"\nSample input:")
print(f"  Prompt: {sample['prompt'][0][:60]}...")
print(f"  Answer: {sample['answer'][0]}")
print(f"  Expected label: {label2id[sample['answer'][0]]}")

# Process it
output = preprocess_function(sample)
print(f"\nPreprocessed output:")
print(f"  Keys: {list(output.keys())}")
print(f"  input_ids shape: {np.array(output['input_ids']).shape}")
print(f"  labels shape: {np.array(output['labels']).shape if 'labels' in output else 'N/A'}")
if 'labels' in output:
    print(f"  labels dtype: {np.array(output['labels']).dtype}")
    print(f"  labels value: {output['labels']}")

# Check if labels match options
print(f"\n✓ Label encoding check:")
print(f"  Original answer: {sample['answer'][0]}")
print(f"  Encoded label: {int(np.array(output['labels'])[0]) if 'labels' in output else 'N/A'}")
print(f"  Correct? {int(np.array(output['labels'])[0]) == label2id[sample['answer'][0]] if 'labels' in output else 'N/A'}")

🧪 TESTING PREPROCESSING FUNCTION

Sample input:
  Prompt: Pick the best possible answer: What is Martin Heidegger's vi...
  Answer: B
  Expected label: 1

Preprocessed output:
  Keys: ['input_ids', 'token_type_ids', 'attention_mask', 'labels']
  input_ids shape: (1, 5, 256)
  labels shape: (1,)
  labels dtype: int64
  labels value: [1]

✓ Label encoding check:
  Original answer: B
  Encoded label: 1
  Correct? True


# Dataset load

In [27]:
from datasets import Dataset

full_dataset = Dataset.from_pandas(df)

encoded_train = full_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=BATCH_SIZE,
    remove_columns=full_dataset.column_names,
)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [28]:
print(encoded_train.features)
print("Label feature dtype:", encoded_train.features["labels"].dtype)

{'input_ids': List(List(Value('int32'))), 'token_type_ids': List(List(Value('int8'))), 'attention_mask': List(List(Value('int8'))), 'labels': Value('int64')}
Label feature dtype: int64


# Training

In [31]:
training_args = TrainingArguments(
    output_dir="./mcq_model",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    # fp16=True,
    dataloader_num_workers=2,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    processing_class=tokenizer,
)
trainer.train()

Step,Training Loss
500,19.813574


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=750, training_loss=13.209049479166667, metrics={'train_runtime': 591.6894, 'train_samples_per_second': 10.14, 'train_steps_per_second': 1.268, 'total_flos': 3946701173760000.0, 'train_loss': 13.209049479166667, 'epoch': 3.0})

# Test predictions & Submission

In [32]:
test_data = Dataset.from_pandas(test_df)
encoded_test = test_data.map(
    preprocess_function,
    batched=True,
    batch_size=100,
    remove_columns=test_data.column_names,
    load_from_cache_file=False
)

preds = trainer.predict(encoded_test)
raw_logits = preds.predictions

raw_logits

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

array([[nan, nan, nan, nan, nan],
       [nan, nan, nan, nan, nan],
       [nan, nan, nan, nan, nan],
       ...,
       [nan, nan, nan, nan, nan],
       [nan, nan, nan, nan, nan],
       [nan, nan, nan, nan, nan]], dtype=float32)

In [33]:
df[:5]

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [ ]:
# Use NumPy directly for top-k extraction to avoid unintended torch dtype casts.
logits = raw_logits.astype(np.float32)
top_3_indices = np.argsort(logits, axis=-1)[:, -3:][:, ::-1]

top_3_predictions = [
    " ".join([id2label[int(idx)] for idx in row])
    for row in top_3_indices
]

top_3_predictions

In [ ]:
top_3_indices = torch.topk(torch.tensor(raw_logits), k=3, dim=-1).indices.numpy()

top_3_predictions = [
    " ".join([id2label[idx] for idx in row])
    for row in top_3_indices
]

top_3_predictions

In [ ]:

test_df["prediction"] = top_3_predictions
submission_df = test_df[["id", "prediction"]].copy()
submission_df.rename(columns={"id": "Id", "prediction": "Prediction"}, inplace=True)
submission_df.to_csv("submission.csv", index=False)